# AE3 – Transformación y enriquecimiento de datos

**Proyecto Integrador III** · Braian Rico · Samuel Rosero · Diego Cuervo

En este notebook partimos del dataset elegido en la EA2, lo **transformamos y validamos por código**, lo **enriquecemos** (variables calculadas y un enriquecimiento con el modelo de lenguaje NVIDIA Nemotron), **respondemos** una tabla por pregunta y **exportamos automáticamente** el dataset preparado, las tablas y un `.txt` descriptivo del DataFrame final.

## 1. Necesidad, preguntas y variables

> **Aviso:** el escenario es **hipotético**, planteado con fines académicos. No existe un cliente real.

**Necesidad.** El equipo de operaciones de un sistema público de bicicletas compartidas necesita saber **cómo cambia la demanda por hora** según la franja horaria, el tipo de día (laboral, fin de semana o festivo), la estación del año y el clima, y **en qué se diferencian los usuarios casuales de los registrados**, para planificar la redistribución de bicicletas y el mantenimiento.

**Pregunta de la EA2:** ¿Cómo varía la cantidad de alquileres por hora según la franja horaria, el tipo de día, la estación y las condiciones del clima, y qué diferencias hay entre usuarios casuales y registrados durante 2011 y 2012?

**Unidad de análisis:** cada fila es **una hora de servicio** del sistema *Capital Bikeshare* (Washington D.C.) con la cantidad de alquileres de esa hora.

### 1.1 Ajuste de las preguntas (justificado)

La pregunta única de la EA2 mezcla cinco dimensiones, y una sola tabla que las cruce a todas sería ilegible. Siguiendo la Tabla 3 del documento de la EA2, la **descomponemos en seis subpreguntas**, cada una con su propia tabla. No cambia el propósito: solo la volvemos respondible por partes.

| Pregunta | Variables que se usan | Resultado esperado (tabla) |
|---|---|---|
| **P1.** ¿En qué horas y franjas se concentra la demanda? | `hora`, `franja_horaria`, `alquileres_total` | Promedio de alquileres por hora del día y por franja, con su participación en el total |
| **P2.** ¿Cómo cambia la demanda según el tipo de día? | `tipo_dia`, `franja_horaria`, `nombre_festivo`, `alquileres_total` | Promedio de alquileres por hora para día laboral, fin de semana y festivo, por franja y por festivo |
| **P3.** ¿Cómo varía entre estaciones y meses? | `estacion`, `nombre_mes`, `alquileres_total` | Promedio y total de alquileres por estación y por mes |
| **P4.** ¿Cómo se relaciona la demanda con el clima? | `clima`, `temp_c`, `rango_temperatura`, `alquileres_total` | Promedio de alquileres por situación climática y por rango de temperatura (°C) |
| **P5.** ¿En qué se diferencian casuales y registrados? | `usuarios_casuales`, `usuarios_registrados`, `tipo_dia`, `franja_horaria`, `estacion`, `tipo_festivo` | Participación (%) de cada tipo de usuario por tipo de día, franja, estación y tipo de festivo |
| **P6.** ¿Cuánto cambió la demanda entre 2011 y 2012? | `anio`, `usuarios_casuales`, `usuarios_registrados`, `alquileres_total` | Totales, promedio por hora y variación (%) por tipo de usuario |

**Limitaciones declaradas en la EA2 que seguimos teniendo en cuenta:** no hay datos por estación de bicicletas ni por viaje; faltan 165 de las 17.544 horas del calendario (0,9 %) sin causa documentada; la situación climática 4 tiene solo 3 registros; el clima viene normalizado; la unidad del viento no está documentada; los datos son de 2011-2012 y de una sola ciudad; y el análisis es **descriptivo, no causal**.

## 2. Preparación del entorno y carga del dataset

Buscamos la raíz del repositorio (la carpeta que contiene `src/clases_po`) y agregamos `src` a la ruta de Python. Las rutas de entrada y salida son **relativas al notebook**.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from clases_po import (
    EnriquecedorFestivos,
    HFDatasetLoader,
    LLMNvidia,
    Reportes,
    TransformacionBike,
)

from clases_po.transformacion_bike import RENOMBRAR_FINAL

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Rutas relativas al notebook (AE3/notebooks/)
RUTA_DATASET = Path("../datos/bike_sharing_preparado.csv")
RUTA_TABLAS = Path("../resultados/tablas")
RUTA_RESUMEN = Path("../resultados/resumen_dataframe_final.txt")
RUTA_LLM = Path("../llm/festivos_nemotron.json")

Cargamos el dataset **completo** con `HFDatasetLoader` y la **revisión fija** de la EA2 (`t22000t/bike-sharing-tabular`, split `train`; copia del [UCI Bike Sharing Dataset, id 275](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) de Fanaee-T y Gama, licencia **CC BY 4.0**). Guardamos `df_original` como **copia sin modificar**: todas las transformaciones trabajan sobre copias, y al final la usamos para comparar *antes* y *después*.

In [2]:
DATASET = "t22000t/bike-sharing-tabular"
SPLIT = "train"
REVISION = "6b645e0cedbc5208fcca11ac94544b088478bf61"

loader = HFDatasetLoader(dataset_name=DATASET, split=SPLIT, revision=REVISION)
df_original = loader.load()
print(f"Filas: {len(df_original):,}".replace(",", ".") + f" | Columnas: {df_original.shape[1]}")
df_original.head()

Filas: 17.379 | Columnas: 17


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 3. Transformación y controles de calidad

Solo transformamos **lo que los datos requieren**. Cada paso indica *qué* cambia, *por qué* y *cuántos registros* afecta. Los pasos viven en la clase `TransformacionBike` (`src/clases_po/transformacion_bike.py`).

### 3.1 Tipos de dato: fecha y fecha-hora

**Qué:** `dteday` llega como texto (`object`). Lo convertimos a `datetime` y lo renombramos `fecha`; además creamos `fecha_hora` = `fecha` + `hr` horas. **Por qué:** sin tipo fecha no se puede ordenar el calendario, calcular el día de la semana ni detectar horas faltantes.

In [3]:
T = TransformacionBike()
df_t = T.convertir_tipos(df_original)

print("Tipo de dteday antes :", df_original["dteday"].dtype)
print("Tipo de fecha después:", df_t["fecha"].dtype)
print("Tipo de fecha_hora   :", df_t["fecha_hora"].dtype)
print("Registros afectados  :", len(df_t), "(todas las filas)")
print("Fechas no convertibles (NaT):", int(df_t["fecha"].isna().sum()))
print("Rango de fechas:", df_t["fecha"].min().date(), "->", df_t["fecha"].max().date())

Tipo de dteday antes : object
Tipo de fecha después: datetime64[ns]
Tipo de fecha_hora   : datetime64[ns]
Registros afectados  : 17379 (todas las filas)
Fechas no convertibles (NaT): 0
Rango de fechas: 2011-01-01 -> 2012-12-31


### 3.2 Controles de calidad

Ejecutamos **todos** los controles y los informamos, incluso los que no detectan nada. La columna `registros_afectados` cuenta las filas (o celdas, en el caso de los nulos) que incumplen la regla. Los dos últimos controles son **informativos**: no son errores de estructura, pero pueden afectar el análisis.

In [4]:
controles = T.controles_calidad(df_t)
problemas = controles[controles["resultado"] == "Problema detectado"]
print(f"Controles ejecutados: {len(controles)}")
print(f"Sin problemas: {(controles['resultado'] == 'Sin problemas').sum()} | "
      f"Problemas detectados: {len(problemas)} | "
      f"Hallazgos informativos: {(controles['resultado'] == 'Hallazgo informativo').sum()}")
controles

Controles ejecutados: 20
Sin problemas: 18 | Problemas detectados: 0 | Hallazgos informativos: 2


,control,regla,registros_afectados,resultado
0,Valores nulos (celdas),Cualquier celda vacía en cualquier columna,0,Sin problemas
1,Filas completamente duplicadas,Dos filas idénticas en todas las columnas,0,Sin problemas
2,Identificador `instant` repetido,El mismo `instant` en más de una fila,0,Sin problemas
3,Fecha y hora repetidas,"Más de una fila con la misma (fecha, hr)",0,Sin problemas
4,cnt distinto de casual + registered,cnt != casual + registered,0,Sin problemas
5,Hora fuera de 0 a 23,hr < 0 o hr > 23,0,Sin problemas
6,Día de la semana incoherente con la fecha,weekday != día de la semana calculado desde fe...,0,Sin problemas
7,Día laboral incoherente,workingday != (lunes a viernes y no festivo),0,Sin problemas
8,Año incoherente con la fecha,yr != año de la fecha - 2011,0,Sin problemas
9,Mes incoherente con la fecha,mnth != mes de la fecha,0,Sin problemas


**Resultado.** Se ejecutaron **20 controles**: **18 no detectaron problemas** (0 registros afectados), **0 detectaron un problema** y **2 son hallazgos informativos**.

- **Sin problemas:** no hay nulos ni duplicados (ni filas completas, ni `instant` repetido, ni fecha y hora repetidas); `cnt` es siempre `casual + registered`; la hora está siempre entre 0 y 23; el día de la semana, el año y el mes son coherentes con la fecha; `workingday` es coherente con el día de la semana y el festivo; los valores normalizados están dentro de [0, 1]; y no hay conteos negativos ni horas con `cnt = 0` (por lo tanto `pct_casual` se puede calcular en todas las filas).
- **Humedad igual a 0:** 22 filas, **todas del 2011-03-10**. Una humedad relativa de 0 % es físicamente improbable y sugiere un dato no registrado. **Decisión: se conservan** (no se eliminan ni se imputan: no tenemos un valor correcto para reemplazarlas y son el 0,13 % de las filas). La humedad no se usa en P1 a P6.
- **Viento igual a 0:** 2.180 filas (12,5 %), repartidas en ambos años (1.107 en 2011 y 1.073 en 2012). El menor valor distinto de cero es 6 (en la escala original), así que hay un salto entre 0 y 6: podría tratarse de viento no medido, pero **no podemos confirmarlo**. **Decisión: se conservan**; el viento no se usa en P1 a P6.

**Registros eliminados, corregidos o imputados por calidad: 0.**

*Nota:* `season` no coincide con el mes calendario (por ejemplo, marzo tiene filas de invierno y de primavera) porque UCI asigna la estación por fecha astronómica. No es un error: por eso P3 mira estación y mes por separado.

### 3.3 Horas faltantes del calendario

**Qué se hace:** construimos el calendario completo de horas entre la primera y la última fecha y buscamos cuáles no tienen fila. **Por qué:** una hora sin fila no es un cero: si la ignoramos sin mirarla, los promedios por hora podrían estar sesgados.

In [5]:
faltantes = T.horas_faltantes(df_t)
por_fecha = T.horas_faltantes_por_fecha(faltantes)

horas_calendario = int((df_t["fecha"].max() - df_t["fecha"].min()).days + 1) * 24
print(f"Horas del calendario : {horas_calendario:,}".replace(",", "."))
print(f"Filas del dataset    : {len(df_t):,}".replace(",", "."))
print(f"Horas faltantes      : {len(faltantes)} ({len(faltantes) / horas_calendario:.1%})")
print(f"Fechas con faltantes : {len(por_fecha)}")
print(f"Fechas con 12 o más horas faltantes: {(por_fecha['horas_faltantes'] >= 12).sum()}")
assert len(faltantes) == horas_calendario - len(df_t) == 165

print("\nFechas con más horas faltantes:")
display(por_fecha.sort_values(["horas_faltantes", "fecha"], ascending=[False, True]).head(8).reset_index(drop=True))

print("Horas faltantes según la hora del día:")
display(faltantes["hora"].value_counts().sort_index().rename("horas_faltantes").to_frame().T)

Horas del calendario : 17.544
Filas del dataset    : 17.379
Horas faltantes      : 165 (0.9%)
Fechas con faltantes : 76
Fechas con 12 o más horas faltantes: 4

Fechas con más horas faltantes:


,fecha,horas_faltantes,horas
0,2012-10-29,23,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
1,2011-01-27,16,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ..."
2,2012-10-30,13,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12"
3,2011-01-18,12,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11"
4,2011-01-26,8,"3, 4, 18, 19, 20, 21, 22, 23"
5,2011-08-28,7,"0, 1, 2, 3, 4, 5, 6"
6,2011-02-22,6,"0, 1, 2, 3, 4, 5"
7,2011-08-27,6,"18, 19, 20, 21, 22, 23"


Horas faltantes según la hora del día:


hora,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
horas_faltantes,5,7,16,34,34,14,6,4,4,4,4,4,3,2,2,2,1,1,3,3,3,3,3,3


**Resultado.** Faltan **165 horas** de las 17.544 del calendario (0,9 %), repartidas en **76 fechas**. Coincide con lo declarado en la EA2. **Cuatro fechas concentran 64 de las 165 horas** (2012-10-29 con 23, 2011-01-27 con 16, 2012-10-30 con 13 y 2011-01-18 con 12); las otras 72 fechas suman 101 horas. Además, **110 de las 165 (66,7 %) caen entre las 0 y las 5 h**, las horas de menor demanda.

**Decisión: no se imputan ni se agregan filas.**

1. No se sabe si son horas **sin alquileres** (un cero real) o horas **sin registro**; la causa no está documentada.
2. Imputar ceros bajaría los promedios de madrugada; imputar valores intermedios inventaría datos. En ambos casos **se sesgarían los promedios**.
3. Es el 0,9 % del calendario.

**Efecto en el análisis:** todos los promedios son "por hora **observada**". Que muchas horas faltantes sean de madrugada es compatible con horas sin alquileres, pero es solo una hipótesis que este dataset no permite comprobar.

### 3.4 Decodificar categorías

**Qué:** reemplazamos los códigos numéricos por etiquetas en español. Las columnas `estacion`, `clima`, `dia_semana`, `festivo` y `dia_laboral` quedan como **categorías ordenadas** (`pd.Categorical(ordered=True)`) y `anio` queda como entero (2011 o 2012), para que las tablas y los gráficos de la AE4 respeten el orden natural (por ejemplo, Invierno → Primavera → Verano → Otoño). **Por qué:** `season = 3` no dice nada; "Verano" sí. **Registros afectados:** las 17.379 filas.

| Columna original | Columna nueva | Regla |
|---|---|---|
| `season` | `estacion` | 1 Invierno, 2 Primavera, 3 Verano, 4 Otoño |
| `weathersit` | `clima` | 1 Despejado o pocas nubes; 2 Niebla con nubes; 3 Lluvia o nieve ligera; 4 Lluvia fuerte, tormenta o nieve con niebla |
| `weekday` | `dia_semana` | 0 = Domingo … 6 = Sábado (verificado contra la fecha en el control 3.2) |
| `yr` | `anio` | 0 → 2011, 1 → 2012 |
| `holiday` | `festivo` | 0 No festivo, 1 Festivo |
| `workingday` | `dia_laboral` | 0 No laboral, 1 Laboral |

In [6]:
df_t = T.decodificar(df_t)

# Cada código debe mapearse a exactamente una etiqueta (sin pérdida de información)
equivalencias = [("season", "estacion"), ("weathersit", "clima"), ("weekday", "dia_semana"),
                 ("yr", "anio"), ("holiday", "festivo"), ("workingday", "dia_laboral")]
for codigo, etiqueta in equivalencias:
    assert df_t.groupby(codigo, observed=True)[etiqueta].nunique().max() == 1
    assert df_t.groupby(etiqueta, observed=True)[codigo].nunique().max() == 1

print("Conteo por etiqueta (el orden es el de la categoría ordenada):")
for _, etiqueta in equivalencias:
    print(f"\n{etiqueta}")
    display(df_t[etiqueta].value_counts(sort=False).rename("filas").to_frame().T)

Conteo por etiqueta (el orden es el de la categoría ordenada):

estacion


estacion,Invierno,Primavera,Verano,Otoño
filas,4242,4409,4496,4232



clima


clima,Despejado o pocas nubes,Niebla con nubes,Lluvia o nieve ligera,"Lluvia fuerte, tormenta o nieve con niebla"
filas,11413,4544,1419,3



dia_semana


dia_semana,Lunes,Martes,Miércoles,Jueves,Viernes,Sábado,Domingo
filas,2479,2453,2475,2471,2487,2512,2502


... (2 outputs left out)


festivo


festivo,No festivo,Festivo
filas,16879,500



dia_laboral


dia_laboral,No laboral,Laboral
filas,5514,11865


**Resultado.** Las seis decodificaciones afectaron a las **17.379 filas** y son **1 a 1** (cada código corresponde a una sola etiqueta y viceversa, comprobado por código), así que no se pierde información. Reparto: 4.242 horas de invierno, 4.409 de primavera, 4.496 de verano y 4.232 de otoño; 8.645 horas de 2011 y 8.734 de 2012; 500 horas festivas y 16.879 no festivas; 11.865 laborales y 5.514 no laborales.

En el clima, 11.413 horas son despejadas, 4.544 con niebla, 1.419 con lluvia o nieve ligera y solo **3** de la situación 4 (0,02 %). **Se conservan** esas 3 filas: son datos reales y eliminarlas no aporta nada; pero **declaramos la limitación**: con 3 horas no se puede concluir nada sobre la situación 4.

### 3.5 Desnormalizar clima a sus unidades originales

**Qué:** el clima viene normalizado (UCI divide por el máximo o por el rango). Volvemos a las unidades originales con las fórmulas documentadas por UCI. **Por qué:** "temperatura = 0,34" no le sirve a operaciones; "7,98 °C" sí, y permite definir rangos de temperatura. **Registros afectados:** las 17.379 filas.

| Variable nueva | Fórmula | Unidad |
|---|---|---|
| `temp_c` | `temp × (39 − (−8)) + (−8)` = `temp × 47 − 8` | °C |
| `sensacion_c` | `atemp × (50 − (−16)) + (−16)` = `atemp × 66 − 16` | °C (sensación térmica) |
| `humedad_pct` | `hum × 100` | % |
| `viento_original` | `windspeed × 67` | **unidad original no documentada por UCI** |

> UCI documenta que `windspeed` se dividió por 67, pero **no dice en qué unidad estaba el viento**. Por eso la llamamos `viento_original` y **no afirmamos que sean km/h**.

In [7]:
df_t = T.desnormalizar(df_t)
comparacion_unidades = pd.DataFrame({
    "variable_original": ["temp", "atemp", "hum", "windspeed"],
    "variable_nueva": ["temp_c", "sensacion_c", "humedad_pct", "viento_original"],
    "mín_original": [df_t[c].min() for c in ["temp", "atemp", "hum", "windspeed"]],
    "máx_original": [df_t[c].max() for c in ["temp", "atemp", "hum", "windspeed"]],
    "mín_nueva": [df_t[c].min() for c in ["temp_c", "sensacion_c", "humedad_pct", "viento_original"]],
    "máx_nueva": [df_t[c].max() for c in ["temp_c", "sensacion_c", "humedad_pct", "viento_original"]],
})
print("Filas afectadas:", len(df_t))
comparacion_unidades

Filas afectadas: 17379


,variable_original,variable_nueva,mín_original,máx_original,mín_nueva,máx_nueva
0,temp,temp_c,0.02,1.0000,-7.06,39.0
1,atemp,sensacion_c,0.00,1.0000,-16.00,50.0
2,hum,humedad_pct,0.00,1.0000,0.00,100.0
3,windspeed,viento_original,0.00,0.8507,0.00,57.0


**Resultado.** Temperatura entre −7,06 y 39,00 °C; sensación térmica entre −16,00 y 50,00 °C; humedad entre 0 y 100 %; viento entre 0 y 57 en la unidad original (sin documentar). Los extremos de la sensación térmica (−16 y 50) y el máximo de la temperatura (39) coinciden con los límites de las fórmulas de UCI (valores normalizados 0 y 1), lo que respalda que la desnormalización es correcta. Las columnas normalizadas originales se quitan más adelante (sección 5.3) porque quedan redundantes.

### 3.6 Comparación intermedia: antes y después de transformar

Todavía conservamos las columnas codificadas junto a las nuevas. Más adelante, tras el enriquecimiento, seleccionamos las columnas pertinentes y repetimos esta comparación con el DataFrame final.

In [8]:
T.comparar_resumen(df_original, df_t)

,métrica,antes,después
0,Filas,17379,17379
1,Columnas,17,28
2,Celdas nulas,0,0
3,Columnas numéricas,16,21
4,Columnas fecha,0,2
5,Columnas categóricas ordenadas,0,5
6,Columnas texto,1,0


## 4. Enriquecimiento con variables derivadas

Agregamos variables calculadas a partir de las que ya tenemos. **No se combina ninguna otra tabla en esta sección** (la combinación con el LLM está en la sección 5).

### 4.1 Franjas horarias: primero miramos dónde está la demanda

Antes de fijar las franjas, miramos el perfil promedio de alquileres por hora del día.

In [9]:
perfil_horario = df_t.groupby("hr")["cnt"].mean().round(1).rename("promedio_alquileres").to_frame().T
display(perfil_horario)
top3 = df_t.groupby("hr")["cnt"].mean().nlargest(3).round(1)
print("Las 3 horas con mayor promedio:", {int(h): float(v) for h, v in top3.items()})

hr,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
promedio_alquileres,53.9,33.4,22.9,11.7,6.4,19.9,76.0,212.1,359.0,219.3,173.7,208.1,253.3,253.7,240.9,251.2,312.0,461.5,425.5,311.5,226.0,172.3,131.3,87.8


Las 3 horas con mayor promedio: {17: 461.5, 18: 425.5, 8: 359.0}


**Lectura del perfil.** La demanda tiene **dos picos**: uno a las **8 h** (359,0 alquileres por hora en promedio) y otro, más alto, a las **17 h (461,5)** y **18 h (425,5)**. La mínima es a las 4 h (6,4). Entre las 2 y las 5 h el promedio es menor a 23; a las 6 h sube a 76,0 y a las 7 h salta a 212,1; de 10 a 15 h se mantiene en una meseta (173,7 a 253,7); de 16 a 19 h está entre 311,5 y 461,5; y de 20 a 23 h baja de 226,0 a 87,8.

**Regla de `franja_horaria`** (definida a partir de ese perfil):

| Franja | Horas | Por qué |
|---|---|---|
| Madrugada | 0-5 h | Demanda mínima (máximo 53,9 a las 0 h) |
| Pico de la mañana | 6-9 h | Subida, pico de las 8 h y bajada |
| Mediodía | 10-15 h | Meseta intermedia |
| Pico de la tarde | 16-19 h | Pico de las 17-18 h con sus hombros |
| Noche | 20-23 h | Descenso hacia la madrugada |

*Limitación:* las franjas se definieron mirando los mismos datos que luego describen; con otros cortes los porcentajes cambiarían un poco.

### 4.2 Crear las variables

Un solo paso crea `nombre_mes`, `franja_horaria`, `tipo_dia`, `rango_temperatura` y `pct_casual`. Después **verificamos** las reglas contra los datos originales.

In [10]:
df_e = T.enriquecer(df_t)

# Verificación 1: tipo_dia coincide con workingday (laboral <-> workingday = 1) y con holiday
cruce = pd.crosstab(df_e["tipo_dia"], df_e["workingday"].map({0: "workingday = 0", 1: "workingday = 1"}))
display(cruce)
assert (cruce.loc["Laboral", "workingday = 0"] == 0 and cruce.loc["Fin de semana", "workingday = 1"] == 0
        and cruce.loc["Festivo", "workingday = 1"] == 0)
assert ((df_e["tipo_dia"] == "Festivo") == (df_e["holiday"] == 1)).all()
print("tipo_dia es coherente con workingday y holiday: 0 inconsistencias.")

# Verificación 2: pct_casual sin nulos (no hay cnt = 0) y dentro de 0-100
assert df_e["pct_casual"].notna().all() and df_e["pct_casual"].between(0, 100).all()
print("pct_casual: sin nulos y entre", df_e["pct_casual"].min(), "y", df_e["pct_casual"].max(), "%")

# Verificación 3: todas las filas tienen franja y rango de temperatura
assert df_e[["franja_horaria", "rango_temperatura", "nombre_mes", "tipo_dia"]].notna().all().all()

# Reparto de filas por variable categórica nueva
for c in ["franja_horaria", "tipo_dia", "rango_temperatura"]:
    display(df_e[c].value_counts(sort=False).rename("filas").to_frame().T)

workingday,workingday = 0,workingday = 1
tipo_dia,,
Laboral,0,11865
Fin de semana,5014,0
Festivo,500,0


tipo_dia es coherente con workingday y holiday: 0 inconsistencias.
pct_casual: sin nulos y entre 0.0 y 100.0 %


franja_horaria,Madrugada (0-5 h),Pico de la mañana (6-9 h),Mediodía (10-15 h),Pico de la tarde (16-19 h),Noche (20-23 h)
filas,4276,2906,4369,2916,2912


tipo_dia,Laboral,Fin de semana,Festivo
filas,11865,5014,500


rango_temperatura,Frío (< 10 °C),Fresco (10 a < 20 °C),Templado (20 a < 30 °C),Caluroso (≥ 30 °C)
filas,5813,5056,5801,709


**Resultado de las verificaciones.**

- `tipo_dia` es **coherente con `workingday` y `holiday`**: las 11.865 horas laborales tienen `workingday = 1`; las 5.014 de fin de semana y las 500 festivas tienen `workingday = 0`; las 500 festivas son exactamente las de `holiday = 1`. **0 inconsistencias.** Las 21 fechas festivas caen de lunes a viernes (son fechas *observadas*), por eso dar prioridad a "Festivo" sobre "Fin de semana" no cambia ninguna fila de fin de semana.
- `pct_casual` **no tiene nulos** y va de 0 a 100 %. Hay 24 horas con 100 % de casuales (0 registrados).
- Reparto de filas por franja: 4.276 (madrugada), 2.906 (pico de la mañana), 4.369 (mediodía), 2.916 (pico de la tarde) y 2.912 (noche). Por rango de temperatura: 5.813 (frío), 5.056 (fresco), 5.801 (templado) y 709 (caluroso).

*Rangos de temperatura:* usamos cortes de 10 °C (10, 20 y 30 °C) porque son fáciles de leer para operaciones y dejan grupos con suficientes horas (el menor, "caluroso", tiene 709).

### 4.3 Tabla de variables incorporadas

| Variable | Fórmula o regla | Significado | Tipo | Unidad | Utilidad (pregunta) |
|---|---|---|---|---|---|
| `fecha` | `dteday` convertido a fecha | Día calendario | `datetime64` | fecha | Todas |
| `fecha_hora` | `fecha` + `hr` horas | Momento exacto de la hora de servicio | `datetime64` | fecha-hora | Detección de horas faltantes |
| `anio` | `yr + 2011` | Año de la observación | entero | año | P6 |
| `nombre_mes` | `mnth` → Enero … Diciembre | Mes con nombre, ordenado | categoría ordenada | - | P3 |
| `dia_semana` | `weekday` → Lunes … Domingo | Día de la semana | categoría ordenada | - | P2 |
| `estacion` | `season` → Invierno … Otoño | Estación del año | categoría ordenada | - | P3, P5 |
| `clima` | `weathersit` → 4 situaciones | Situación climática | categoría ordenada | - | P4 |
| `temp_c` | `temp × 47 − 8` | Temperatura | decimal | °C | P4 |
| `sensacion_c` | `atemp × 66 − 16` | Sensación térmica | decimal | °C | Referencia |
| `humedad_pct` | `hum × 100` | Humedad relativa | decimal | % | Referencia |
| `viento_original` | `windspeed × 67` | Velocidad del viento | decimal | sin documentar | Referencia |
| `franja_horaria` | `hr` 0-5 Madrugada; 6-9 Pico de la mañana; 10-15 Mediodía; 16-19 Pico de la tarde; 20-23 Noche | Momento del día según el perfil de demanda | categoría ordenada | - | P1, P2, P5 |
| `tipo_dia` | `Festivo` si `holiday = 1`; `Fin de semana` si sábado o domingo; `Laboral` en otro caso | Tipo de día | categoría ordenada | - | P2, P5 |
| `rango_temperatura` | `temp_c` < 10 Frío; 10 a < 20 Fresco; 20 a < 30 Templado; ≥ 30 Caluroso | Rango de temperatura | categoría ordenada | °C | P4 |
| `pct_casual` | `casual / cnt × 100` | Participación de casuales en esa hora | decimal | % | P5 |
| `nombre_festivo` | Unión por fecha con la clasificación de festivos (sección 5) | Qué festivo es; "No festivo" si no lo es | texto | - | P2 |
| `tipo_festivo` | Ídem | Federal o local de D.C.; "No festivo" si no lo es | texto | - | P5 |

Las columnas originales que se conservan se renombran para mayor claridad: `hr` → `hora`, `mnth` → `mes`, `casual` → `usuarios_casuales`, `registered` → `usuarios_registrados`, `cnt` → `alquileres_total`.

## 5. Enriquecimiento con un LLM: festivos con NVIDIA Nemotron

**Necesidad:** el dataset marca los días festivos con `holiday = 1`, pero **no dice cuál festivo es** ni si es **federal** o **local de Washington D.C.** Con esa información podemos mirar la demanda festivo por festivo (P2) y separar el tipo de festivo (P5).

**Qué se agrega:** dos columnas, `nombre_festivo` y `tipo_festivo`. **No se inventan filas ni valores de demanda.**

**Cómo funciona** (clases `LLMNvidia` y `EnriquecedorFestivos`):

1. Se toman las **fechas únicas** con `holiday = 1`.
2. **Caché (`AE3/llm/festivos_nemotron.json`):** si el archivo existe, **no se vuelve a llamar al LLM**. Si no existe y hay `NVIDIA_API_KEY`, se llama a Nemotron (`nvidia/nemotron-3-ultra-550b-a55b`), se valida la respuesta y se guarda el archivo con metadatos (modelo, fecha, prompt). Si no existe y no hay clave, o el LLM falla, se usa un **catálogo de respaldo** y **no** se escribe el archivo del LLM.
3. **Catálogo de respaldo:** feriados federales de EE. UU. según la [OPM](https://www.opm.gov/policy-data-oversight/pay-leave/federal-holidays/) (fechas *observadas*) más el Día de la Emancipación de D.C. (local).
4. Se **validan** las fechas y los nombres, y se une al dataset por fecha comprobando que **no cambien las filas ni la suma de alquileres**.

La clave del API nunca se imprime ni se guarda: se lee de la variable de entorno `NVIDIA_API_KEY`.

In [11]:
llm = LLMNvidia()  # lee NVIDIA_API_KEY del entorno (si no existe, queda inactivo)
print(llm)

enriquecedor = EnriquecedorFestivos(RUTA_LLM, llm=llm)
fechas_festivas = enriquecedor.fechas_festivas(df_e)
tabla_festivos = enriquecedor.obtener(fechas_festivas)

print()
print("=" * 70)
print(f"FUENTE DE LOS FESTIVOS: {enriquecedor.fuente.upper()}")
print(f"Detalle: {enriquecedor.detalle_fuente}")
print("=" * 70)
print(f"Fechas únicas con holiday = 1 en el dataset: {len(fechas_festivas)}")
tabla_festivos

LLMNvidia(modelo='nvidia/nemotron-3-ultra-550b-a55b', pensar=False, estado=sin conexión al LLM: se usará el archivo guardado o, si no existe, el respaldo)

FUENTE DE LOS FESTIVOS: LLM
Detalle: archivo versionado festivos_nemotron.json (modelo nvidia/nemotron-3-ultra-550b-a55b, generado el 2026-11-07T19:54:40+00:00); no se llamó al LLM
Fechas únicas con holiday = 1 en el dataset: 21


,fecha,nombre_festivo,tipo_festivo
0,2011-01-17,Día de Martin Luther King Jr.,Federal
1,2011-02-21,Cumpleaños de Washington,Federal
2,2011-04-15,Día de la Emancipación de D.C.,Local (D.C.)
3,2011-05-30,Día de los Caídos,Federal
4,2011-07-04,Día de la Independencia,Federal
5,2011-09-05,Día del Trabajo,Federal
6,2011-10-10,Día de Colón,Federal
7,2011-11-11,Día de los Veteranos,Federal
8,2011-11-24,Acción de Gracias,Federal
9,2011-12-26,Navidad,Federal


### 5.1 Validación de la clasificación

Primero, el conjunto de fechas clasificadas debe coincidir **exactamente** con las fechas `holiday = 1` del dataset: ni faltantes ni sobrantes.

In [12]:
validacion = enriquecedor.validar_fechas(fechas_festivas, tabla_festivos)
print("Fechas del dataset sin clasificar (faltantes):", validacion["faltantes"])
print("Fechas clasificadas que no son festivo en el dataset (sobrantes):", validacion["sobrantes"])
assert not validacion["faltantes"] and not validacion["sobrantes"]
assert tabla_festivos["fecha"].is_unique
print("Las fechas coinciden exactamente:", len(tabla_festivos), "de", len(fechas_festivas))

Fechas del dataset sin clasificar (faltantes): []
Fechas clasificadas que no son festivo en el dataset (sobrantes): []
Las fechas coinciden exactamente: 21 de 21


Segundo, comparamos, fecha a fecha, el nombre y el tipo del LLM contra el catálogo oficial de la OPM.

> Cuando la fuente es el **respaldo**, esta comparación es trivial (se compara el catálogo consigo mismo). Se vuelve informativa cuando la fuente es el **LLM**: ahí muestra si Nemotron acertó cada fecha.

In [13]:
comparacion_llm = enriquecedor.comparar_con_catalogo(tabla_festivos)
print(f"Fuente comparada: {enriquecedor.fuente}")
print(f"Coinciden nombre: {int(comparacion_llm['coincide_nombre'].sum())} de {len(comparacion_llm)}")
print(f"Coinciden tipo  : {int(comparacion_llm['coincide_tipo'].sum())} de {len(comparacion_llm)}")
comparacion_llm

Fuente comparada: LLM
Coinciden nombre: 21 de 21
Coinciden tipo  : 21 de 21


,fecha,nombre_festivo,nombre_opm,tipo_festivo,tipo_opm,coincide_nombre,coincide_tipo
0,2011-01-17,Día de Martin Luther King Jr.,Día de Martin Luther King Jr.,Federal,Federal,True,True
1,2011-02-21,Cumpleaños de Washington,Cumpleaños de Washington,Federal,Federal,True,True
2,2011-04-15,Día de la Emancipación de D.C.,Día de la Emancipación de D.C.,Local (D.C.),Local (D.C.),True,True
3,2011-05-30,Día de los Caídos,Día de los Caídos,Federal,Federal,True,True
4,2011-07-04,Día de la Independencia,Día de la Independencia,Federal,Federal,True,True
5,2011-09-05,Día del Trabajo,Día del Trabajo,Federal,Federal,True,True
6,2011-10-10,Día de Colón,Día de Colón,Federal,Federal,True,True
7,2011-11-11,Día de los Veteranos,Día de los Veteranos,Federal,Federal,True,True
8,2011-11-24,Acción de Gracias,Acción de Gracias,Federal,Federal,True,True
9,2011-12-26,Navidad,Navidad,Federal,Federal,True,True


Tercero, el control más importante para el dataset: **cruzamos el catálogo oficial de la OPM con las fechas que el dataset marca como festivo**, en ambos sentidos. Así vemos si el dataset marca algún día que la OPM no lista, o si omite alguno que sí.

In [14]:
cruce_opm = enriquecedor.comparar_catalogo_con_dataset(fechas_festivas)
solo_dataset = cruce_opm[cruce_opm["marcada_en_dataset"] & ~cruce_opm["en_catalogo_opm"]]
solo_opm = cruce_opm[~cruce_opm["marcada_en_dataset"] & cruce_opm["en_catalogo_opm"]]
print(f"Fechas del catálogo OPM (2011-2012, fechas observadas): {int(cruce_opm['en_catalogo_opm'].sum())}")
print(f"Fechas con holiday = 1 en el dataset: {int(cruce_opm['marcada_en_dataset'].sum())}")
print(f"Marcadas en el dataset pero NO en la OPM: {len(solo_dataset)}")
print(f"En la OPM pero NO marcadas en el dataset: {len(solo_opm)}")
cruce_opm

Fechas del catálogo OPM (2011-2012, fechas observadas): 21
Fechas con holiday = 1 en el dataset: 21
Marcadas en el dataset pero NO en la OPM: 0
En la OPM pero NO marcadas en el dataset: 0


,fecha,marcada_en_dataset,en_catalogo_opm,festivo_opm
0,2011-01-17,True,True,Día de Martin Luther King Jr.
1,2011-02-21,True,True,Cumpleaños de Washington
2,2011-04-15,True,True,Día de la Emancipación de D.C.
3,2011-05-30,True,True,Día de los Caídos
4,2011-07-04,True,True,Día de la Independencia
5,2011-09-05,True,True,Día del Trabajo
6,2011-10-10,True,True,Día de Colón
7,2011-11-11,True,True,Día de los Veteranos
8,2011-11-24,True,True,Acción de Gracias
9,2011-12-26,True,True,Navidad


**Resultado.** El dataset marca **21 fechas únicas** como festivo (500 horas observadas). El catálogo oficial de la OPM, con fechas observadas, tiene **21 fechas** para 2011-2012 y **coinciden exactamente**: 0 fechas marcadas en el dataset que la OPM no liste y 0 fechas de la OPM que el dataset no marque. **No hay discrepancias que reportar.** De las 21, 19 son federales y 2 son locales de D.C. (Día de la Emancipación: 2011-04-15 y 2012-04-16). El Año Nuevo de 2011 (sábado 1 de enero) se observó el 31 de diciembre de 2010, fuera del dataset, y por eso en 2011 no aparece.

**Sobre la fuente.** En esta ejecución la línea `FUENTE DE LOS FESTIVOS` indica `LLM`: la fuente fue el archivo versionado `AE3/llm/festivos_nemotron.json`, generado con Nemotron; sus 21 fechas coinciden en nombre y tipo con el calendario oficial de la OPM. Si el archivo no existe, la celda llama al LLM (si hay clave) o usa el catálogo de respaldo (`respaldo`).

### 5.2 Unión al dataset

Unimos por `fecha` con un *left join* (`validate="many_to_one"`: cada fecha del dataset puede tener a lo sumo una fila en la tabla de festivos). La función falla si la unión cambia el número de filas o la suma de alquileres.

**Decisión sobre las filas sin festivo:** reciben `"No festivo"` en `nombre_festivo` y `tipo_festivo`, y **no** nulos. Es una categoría real (ausencia de festivo), no un dato faltante, y así los agrupamientos no descartan esas filas.

In [15]:
filas_antes = len(df_e)
suma_antes = int(df_e["cnt"].sum())
df_unido = EnriquecedorFestivos.unir(df_e, tabla_festivos, col_suma="cnt")

print(f"Filas antes de la unión   : {filas_antes}")
print(f"Filas después de la unión : {len(df_unido)}")
print(f"Suma de alquileres antes  : {suma_antes:,}".replace(",", "."))
print(f"Suma de alquileres después: {int(df_unido['cnt'].sum()):,}".replace(",", "."))
assert len(df_unido) == filas_antes and int(df_unido["cnt"].sum()) == suma_antes

# Coherencia: las filas con nombre de festivo son exactamente las de tipo_dia = Festivo
con_festivo = df_unido["nombre_festivo"] != "No festivo"
assert (con_festivo == (df_unido["tipo_dia"] == "Festivo")).all()
print(f"Filas con festivo: {int(con_festivo.sum())} | sin festivo: {int((~con_festivo).sum())}")
display(df_unido["tipo_festivo"].value_counts().rename("filas").to_frame().T)

Filas antes de la unión   : 17379
Filas después de la unión : 17379
Suma de alquileres antes  : 3.292.679
Suma de alquileres después: 3.292.679
Filas con festivo: 500 | sin festivo: 16879


tipo_festivo,No festivo,Federal,Local (D.C.)
filas,16879,452,48


### 5.3 Selección de columnas y DataFrame final

**Qué se conserva:** el identificador `instant`, la fecha y la hora, las variables decodificadas y desnormalizadas, las derivadas, las dos de festivos y los tres conteos de alquileres.

**Qué se quita y por qué:** las columnas originales codificadas o normalizadas (`season`, `weathersit`, `weekday`, `yr`, `temp`, `atemp`, `hum`, `windspeed`) quedan **redundantes** porque su información está, sin pérdida, en las columnas decodificadas y desnormalizadas (lo comprobamos en 3.4); dejarlas sólo invitaría a usar por error la versión normalizada. `holiday` y `workingday` también se quitan: `tipo_dia` los resume y vimos en 4.2 que es coherente con ambos. `instant` se **conserva** como identificador único de la hora (sin duplicados, 3.2).

**No se elimina ninguna fila.**

In [16]:
df_final = T.seleccionar_columnas(df_unido)
eliminadas = [c for c in df_unido.columns if c not in df_final.columns and c not in RENOMBRAR_FINAL]
print(f"Columnas eliminadas ({len(eliminadas)}): {eliminadas}")
print(f"DataFrame final: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
df_final.head()

Columnas eliminadas (12): ['season', 'yr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'festivo', 'dia_laboral']
DataFrame final: 17379 filas x 23 columnas


,instant,fecha,fecha_hora,anio,mes,nombre_mes,dia_semana,hora,franja_horaria,tipo_dia,nombre_festivo,tipo_festivo,estacion,clima,temp_c,sensacion_c,rango_temperatura,humedad_pct,viento_original,usuarios_casuales,usuarios_registrados,alquileres_total,pct_casual
0,1,2011-01-01,2011-01-01 00:00:00,2011,1,Enero,Sábado,0,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,3.28,3.0,Frío (< 10 °C),81.0,0.0,3,13,16,18.75
1,2,2011-01-01,2011-01-01 01:00:00,2011,1,Enero,Sábado,1,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,2.34,2.0,Frío (< 10 °C),80.0,0.0,8,32,40,20.00
2,3,2011-01-01,2011-01-01 02:00:00,2011,1,Enero,Sábado,2,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,2.34,2.0,Frío (< 10 °C),80.0,0.0,5,27,32,15.62
3,4,2011-01-01,2011-01-01 03:00:00,2011,1,Enero,Sábado,3,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,3.28,3.0,Frío (< 10 °C),75.0,0.0,3,10,13,23.08
4,5,2011-01-01,2011-01-01 04:00:00,2011,1,Enero,Sábado,4,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,3.28,3.0,Frío (< 10 °C),75.0,0.0,0,1,1,0.00


### 5.4 Comparación final: antes y después

Comparamos `df_original` (copia sin modificar) con `df_final`: filas, columnas, tipos y nulos.

In [17]:
assert df_original.shape == (17379, 17) and df_original["dteday"].dtype == object  # el original no se tocó
display(T.comparar_resumen(df_original, df_final))
T.comparar_columnas(df_original, df_final)

,métrica,antes,después
0,Filas,17379,17379
1,Columnas,17,23
2,Celdas nulas,0,0
3,Columnas numéricas,16,12
4,Columnas fecha,0,2
5,Columnas categóricas ordenadas,0,7
6,Columnas texto,1,2


,columna,tipo_antes,nulos_antes,tipo_después,nulos_después
0,instant,int64,0,int64,0
1,dteday,object,0,-,-
2,season,int64,0,-,-
3,yr,int64,0,-,-
4,mnth,int64,0,-,-
5,hr,int64,0,-,-
6,holiday,int64,0,-,-
7,weekday,int64,0,-,-
8,workingday,int64,0,-,-
9,weathersit,int64,0,-,-


**Resultado.** Filas: **17.379 antes y después** (0 eliminadas, 0 agregadas). Columnas: **17 → 23**. Nulos: **0 → 0**. Tipos: antes 16 numéricas y 1 de texto; después 12 numéricas, 2 de fecha, 7 categorías ordenadas y 2 de texto (`nombre_festivo` y `tipo_festivo`). La suma de alquileres no cambió (3.292.679).

De las 17 columnas originales: 1 se conserva igual (`instant`), 5 se conservan con otro nombre (`hr`, `mnth`, `casual`, `registered`, `cnt`) y 11 se quitan por redundantes (`dteday`, `season`, `yr`, `holiday`, `weekday`, `workingday`, `weathersit`, `temp`, `atemp`, `hum`, `windspeed`). Se incorporan 17 columnas nuevas.

## 6. Respuestas a las preguntas

**Convenciones de todas las tablas:**

- *Promedio de alquileres por hora* = promedio de `alquileres_total` sobre las **horas de servicio observadas** (unidad: alquileres por hora). Las 165 horas sin registro no entran en ningún promedio.
- *Total de alquileres* = suma de `alquileres_total` (unidad: alquileres) en 2011-2012.
- `pct_del_total_alquileres` = participación del grupo en el total de alquileres (%). Dentro de cada tabla suma 100 % (con una diferencia de ±0,01 por el redondeo a 2 decimales).
- Nivel de detalle: cada fila de una tabla es un **grupo** (hora, franja, estación…), no una hora de servicio.
- Todo es **descriptivo**: describe qué pasó, no por qué.

In [18]:
TOTAL_ALQUILERES = int(df_final["alquileres_total"].sum())
TABLAS = {}  # nombre del archivo -> DataFrame


def resumen_demanda(df, por):
    """Resumen de la demanda por grupo (una fila por grupo)."""
    g = df.groupby(por, observed=True)
    tabla = g.agg(
        horas_observadas=("alquileres_total", "size"),
        total_alquileres=("alquileres_total", "sum"),
        promedio_alquileres_por_hora=("alquileres_total", "mean"),
        mediana_alquileres_por_hora=("alquileres_total", "median"),
    ).reset_index()
    tabla["pct_del_total_alquileres"] = tabla["total_alquileres"] / df["alquileres_total"].sum() * 100
    return tabla.round({"promedio_alquileres_por_hora": 2, "mediana_alquileres_por_hora": 1, "pct_del_total_alquileres": 2})


def participacion(df, por):
    """Participación (%) de casuales y registrados dentro de cada grupo."""
    g = df.groupby(por, observed=True)[["usuarios_casuales", "usuarios_registrados", "alquileres_total"]].sum().reset_index()
    g["pct_casuales"] = g["usuarios_casuales"] / g["alquileres_total"] * 100
    g["pct_registrados"] = g["usuarios_registrados"] / g["alquileres_total"] * 100
    return g.round({"pct_casuales": 2, "pct_registrados": 2})

### P1. ¿En qué horas y franjas se concentra la demanda?

**Variables:** `hora`, `franja_horaria`, `alquileres_total`. **Nivel de detalle:** una fila por franja (y otra tabla con una fila por hora del día).

In [19]:
p1_franja = resumen_demanda(df_final, "franja_horaria")
p1_hora = resumen_demanda(df_final, ["hora", "franja_horaria"])
TABLAS["P1_demanda_por_franja_horaria"] = p1_franja
TABLAS["P1_demanda_por_hora_del_dia"] = p1_hora
display(p1_franja)
display(p1_hora)

,franja_horaria,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,Madrugada (0-5 h),4276,106509,24.91,13.0,3.23
1,Pico de la mañana (6-9 h),2906,629742,216.70,169.0,19.13
2,Mediodía (10-15 h),4369,1005711,230.19,202.0,30.54
3,Pico de la tarde (16-19 h),2916,1101169,377.63,361.0,33.44
4,Noche (20-23 h),2912,449548,154.38,136.0,13.65


,hora,franja_horaria,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,0,Madrugada (0-5 h),726,39130,53.90,40.0,1.19
1,1,Madrugada (0-5 h),724,24164,33.38,20.0,0.73
2,2,Madrugada (0-5 h),715,16352,22.87,11.0,0.50
3,3,Madrugada (0-5 h),697,8174,11.73,6.0,0.25
4,4,Madrugada (0-5 h),697,4428,6.35,6.0,0.13
5,5,Madrugada (0-5 h),717,14261,19.89,19.0,0.43
6,6,Pico de la mañana (6-9 h),725,55132,76.04,76.0,1.67
7,7,Pico de la mañana (6-9 h),727,154171,212.06,208.0,4.68
8,8,Pico de la mañana (6-9 h),727,261001,359.01,385.0,7.93
9,9,Pico de la mañana (6-9 h),727,159438,219.31,216.0,4.84


**Hallazgo.** La demanda se concentra en las dos franjas pico: la **tarde (16-19 h)** tiene el mayor promedio, **377,63 alquileres por hora**, y reúne el **33,44 %** de todos los alquileres con solo 4 de las 24 horas del día. Le siguen mediodía (230,19 por hora; 30,54 %, en 6 horas), pico de la mañana (216,70; 19,13 %), noche (154,38; 13,65 %) y madrugada (24,91; 3,23 %). Por hora, el máximo es a las **17 h (461,45)**, luego las **18 h (425,51)** y las **8 h (359,01)**; el mínimo es a las 4 h (6,35). Las 17 y 18 h juntas son el 19,64 % de todos los alquileres.

**Utilidad para operaciones.** Indica *cuándo* reforzar la redistribución (antes de las 8 h y antes de las 16-17 h) y *cuándo* conviene hacer mantenimiento (de 2 a 5 h, con menos de 23 alquileres por hora).

**Limitaciones.** Es un promedio de dos años y de todas las estaciones juntas; sin datos por estación de bicicletas no sabemos *dónde* hacer la redistribución; las 165 horas sin registro no entran en los promedios; es descriptivo, no explica por qué.

### P2. ¿Cómo cambia la demanda según el tipo de día?

**Variables:** `tipo_dia`, `franja_horaria`, `nombre_festivo`, `alquileres_total`. **Nivel de detalle:** una fila por tipo de día; otra por tipo de día y franja; y otra por festivo.

Hay muchas más horas de día laboral que de festivo (los festivos son 21 fechas en dos años), por eso comparamos **promedios por hora**, no totales.

In [20]:
p2_tipo = resumen_demanda(df_final, "tipo_dia")
p2_tipo_franja = resumen_demanda(df_final, ["tipo_dia", "franja_horaria"])
TABLAS["P2_demanda_por_tipo_de_dia"] = p2_tipo
TABLAS["P2_demanda_por_tipo_de_dia_y_franja"] = p2_tipo_franja
display(p2_tipo)
display(p2_tipo_franja.pivot(index="franja_horaria", columns="tipo_dia", values="promedio_alquileres_por_hora").rename_axis(columns="promedio por hora"))

,tipo_dia,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,Laboral,11865,2292410,193.21,151.0,69.62
1,Fin de semana,5014,921834,183.85,122.0,28.00
2,Festivo,500,78435,156.87,97.0,2.38


promedio por hora,Laboral,Fin de semana,Festivo
franja_horaria,,,
Madrugada (0-5 h),16.44,44.72,25.48
Pico de la mañana (6-9 h),277.91,82.19,113.04
Mediodía (10-15 h),179.66,346.36,264.85
Pico de la tarde (16-19 h),414.75,300.68,263.99
Noche (20-23 h),165.78,130.43,122.45


**Hallazgo.** En promedio, un día laboral tiene **193,21 alquileres por hora**, un fin de semana **183,85** y un festivo **156,87** (el festivo es 18,8 % menor que el laboral). Las diferencias globales son moderadas; **lo que cambia es el perfil a lo largo del día**:

- **Día laboral:** dos picos, mañana (**277,91**) y tarde (**414,75**), con poco uso al mediodía (179,66).
- **Fin de semana:** casi sin pico de mañana (82,19) y el mayor uso en el **mediodía (346,36)**, que casi duplica el del día laboral (179,66).
- **Festivo:** perfil parecido al fin de semana, con mediodía (264,85) y tarde (263,99) similares y mañana baja (113,04).
- En la **madrugada** el fin de semana (44,72) tiene más de 2,7 veces el uso del día laboral (16,44).

**Utilidad.** Los días laborales requieren reforzar antes de los picos de desplazamiento; los fines de semana y festivos, el mediodía y la tarde. Las ventanas de mantenimiento son distintas según el tipo de día.

**Limitaciones.** El festivo se calcula con 500 horas (21 fechas) y el fin de semana con 5.014; el clima y la estación no están controlados (un festivo en invierno pesa distinto que uno en verano).

**Por festivo.** Para cada festivo (agrupando las fechas de 2011 y 2012 que comparten nombre) mostramos cuántas fechas hay, la demanda promedio por hora y el porcentaje de casuales. Como cada festivo tiene una o dos fechas, **son muestras pequeñas**.

In [21]:
festivos = df_final[df_final["nombre_festivo"] != "No festivo"]
p2_festivo = (
    festivos.groupby(["nombre_festivo", "tipo_festivo"])
    .agg(
        fechas=("fecha", "nunique"),
        horas_observadas=("alquileres_total", "size"),
        total_alquileres=("alquileres_total", "sum"),
        promedio_alquileres_por_hora=("alquileres_total", "mean"),
        total_casuales=("usuarios_casuales", "sum"),
    )
    .reset_index()
)
p2_festivo["pct_casuales"] = p2_festivo["total_casuales"] / p2_festivo["total_alquileres"] * 100
p2_festivo = (
    p2_festivo.drop(columns="total_casuales")
    .round({"promedio_alquileres_por_hora": 2, "pct_casuales": 2})
    .sort_values("promedio_alquileres_por_hora", ascending=False)
    .reset_index(drop=True)
)
TABLAS["P2_demanda_por_festivo"] = p2_festivo
promedio_laboral = df_final.loc[df_final["tipo_dia"] == "Laboral", "alquileres_total"].mean()
print(f"Referencia: promedio en día laboral = {promedio_laboral:.2f} alquileres por hora")
print(f"Fuente de los festivos: {enriquecedor.fuente}")
p2_festivo

Referencia: promedio en día laboral = 193.21 alquileres por hora
Fuente de los festivos: LLM


,nombre_festivo,tipo_festivo,fechas,horas_observadas,total_alquileres,promedio_alquileres_por_hora,pct_casuales
0,Día de la Independencia,Federal,2,48,13446,280.12,41.85
1,Día de Colón,Federal,2,48,10595,220.73,22.54
2,Día de los Caídos,Federal,2,48,10141,211.27,40.49
3,Día de los Veteranos,Federal,2,48,9637,200.77,15.95
4,Día de la Emancipación de D.C.,Local (D.C.),2,48,9496,197.83,19.38
5,Día del Trabajo,Federal,2,48,9385,195.52,34.11
6,Cumpleaños de Washington,Federal,2,47,4236,90.13,16.45
7,Año Nuevo,Federal,1,23,1951,84.83,12.51
8,Acción de Gracias,Federal,2,48,3920,81.67,38.65
9,Día de Martin Luther King Jr.,Federal,2,48,3298,68.71,10.13


**Hallazgo.** El festivo con mayor demanda promedio es el **Día de la Independencia (280,12 por hora)**, seguido de Día de Colón (220,73) y Día de los Caídos (211,27); los de menor demanda son **Navidad (50,65)**, Día de Martin Luther King Jr. (68,71), Acción de Gracias (81,67), Año Nuevo (84,83) y Cumpleaños de Washington (90,13). Como referencia, el promedio de un día laboral es 193,21. El porcentaje de casuales es mayor en Día de la Independencia (41,85 %) y Día de los Caídos (40,49 %) y menor en Martin Luther King Jr. (10,13 %).

**Cuidado al interpretar.** Los tres festivos de mayor demanda caen entre mayo y octubre y los cinco de menor demanda en meses fríos (enero, febrero, noviembre y diciembre): la tabla mezcla el efecto del festivo con el de la estación y el clima, así que **no se puede atribuir la diferencia al festivo en sí**. Además, cada festivo tiene solo una o dos fechas (de 23 a 48 horas): son **muestras pequeñas** y el orden es solo indicativo.

**Utilidad.** Permite anticipar que los festivos de verano se parecen a un fin de semana de alta demanda, y los de invierno a un día de poca demanda. **Limitación:** la clasificación del festivo (nombre y tipo) depende de la fuente indicada arriba (LLM o respaldo OPM).

### P3. ¿Cómo varía la demanda entre estaciones y meses?

**Variables:** `estacion`, `nombre_mes`, `alquileres_total`. **Nivel de detalle:** una fila por estación y una por mes (ambos años juntos).

In [22]:
p3_estacion = resumen_demanda(df_final, "estacion")
p3_mes = resumen_demanda(df_final, ["mes", "nombre_mes"])
TABLAS["P3_demanda_por_estacion"] = p3_estacion
TABLAS["P3_demanda_por_mes"] = p3_mes
display(p3_estacion)
display(p3_mes)

# Solo para interpretar (no se exporta): promedio por hora de cada estación en cada año
print("Promedio de alquileres por hora, por estación y año:")
display(df_final.groupby(["estacion", "anio"], observed=True)["alquileres_total"].mean().round(1).unstack())

,estacion,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,Invierno,4242,471348,111.11,76.0,14.32
1,Primavera,4409,918589,208.34,165.0,27.90
2,Verano,4496,1061129,236.02,199.0,32.23
3,Otoño,4232,841613,198.87,155.5,25.56


,mes,nombre_mes,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,1,Enero,1429,134933,94.42,66.0,4.10
1,2,Febrero,1341,151352,112.87,82.0,4.60
2,3,Marzo,1473,228920,155.41,104.0,6.95
3,4,Abril,1437,269094,187.26,136.0,8.17
4,5,Mayo,1488,331686,222.91,188.5,10.07
5,6,Junio,1440,346342,240.52,203.0,10.52
6,7,Julio,1488,344948,231.82,202.0,10.48
7,8,Agosto,1475,351194,238.10,204.0,10.67
8,9,Septiembre,1437,345991,240.77,190.0,10.51
9,10,Octubre,1451,322352,222.16,174.0,9.79


Promedio de alquileres por hora, por estación y año:


anio,2011,2012
estacion,,
Invierno,72.5,147.8
Primavera,157.7,259.0
Verano,187.3,284.3
Otoño,152.8,245.7


**Hallazgo.** El **verano** tiene el mayor promedio (**236,02 por hora**; 32,23 % del total), seguido de primavera (208,34), otoño (198,87) e **invierno (111,11)**, que es menos de la mitad que el verano. Por mes, el promedio más alto es septiembre (240,77), junio (240,52) y agosto (238,10), y el más bajo es enero (94,42), febrero (112,87) y diciembre (142,30). En totales, agosto suma más (351.194 alquileres; 10,67 %) y enero menos (134.933; 4,10 %).

**Cuidado al interpretar.** El crecimiento de 2011 a 2012 (ver P6) se mezcla con la estacionalidad: por ejemplo, el promedio del invierno fue 72,5 en 2011 y 147,8 en 2012, y el del verano 187,3 y 284,3. Además, algunos meses mezclan estaciones (marzo, junio, septiembre y diciembre), por eso mostramos ambas tablas.

**Utilidad.** Orienta cuándo dimensionar la flota (junio a septiembre) y cuándo concentrar el mantenimiento mayor (diciembre a febrero). **Limitaciones.** Solo dos años; una sola ciudad; no se controla el clima, que también cambia con la estación.

### P4. ¿Cómo se relaciona la demanda con el clima?

**Variables:** `clima`, `temp_c`, `rango_temperatura`, `alquileres_total`. **Nivel de detalle:** una fila por situación climática y una por rango de temperatura (°C).

In [23]:
p4_clima = resumen_demanda(df_final, "clima")
p4_temp = resumen_demanda(df_final, "rango_temperatura")
TABLAS["P4_demanda_por_situacion_climatica"] = p4_clima
TABLAS["P4_demanda_por_rango_de_temperatura"] = p4_temp
display(p4_clima)
display(p4_temp)

,clima,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,Despejado o pocas nubes,11413,2338173,204.87,159.0,71.01
1,Niebla con nubes,4544,795952,175.17,133.0,24.17
2,Lluvia o nieve ligera,1419,158331,111.58,63.0,4.81
3,"Lluvia fuerte, tormenta o nieve con niebla",3,223,74.33,36.0,0.01


,rango_temperatura,horas_observadas,total_alquileres,promedio_alquileres_por_hora,mediana_alquileres_por_hora,pct_del_total_alquileres
0,Frío (< 10 °C),5813,626084,107.70,71.0,19.01
1,Fresco (10 a < 20 °C),5056,949036,187.70,148.0,28.82
2,Templado (20 a < 30 °C),5801,1486229,256.20,220.0,45.14
3,Caluroso (≥ 30 °C),709,231330,326.28,295.0,7.03


**Hallazgo.** El promedio es mayor con **clima despejado o pocas nubes (204,87 por hora)**, baja con niebla y nubes (175,17) y cae a **111,58** con lluvia o nieve ligera, un 45,5 % menos que con el cielo despejado. La situación 4 (lluvia fuerte, tormenta) tiene un promedio de 74,33, pero con **solo 3 horas observadas (223 alquileres)**: no se puede concluir nada. Por temperatura, el promedio crece en cada rango: **107,70** (< 10 °C), **187,70** (10 a < 20 °C), **256,20** (20 a < 30 °C) y **326,28** (≥ 30 °C, 709 horas).

**Utilidad.** Permite anticipar menor demanda en días de frío o de lluvia ligera, y mayor demanda con calor.

**Limitaciones.** La temperatura está muy ligada a la estación y a la hora del día, así que esta relación **no es causal** ni está ajustada por esos factores; el clima viene normalizado y fue desnormalizado con las fórmulas de UCI; la situación 4 tiene solo 3 horas.

### P5. ¿En qué se diferencian casuales y registrados?

**Variables:** `usuarios_casuales`, `usuarios_registrados`, `tipo_dia`, `franja_horaria`, `estacion`, `tipo_festivo`. **Nivel de detalle:** una fila por categoría de cada dimensión (formato largo: columna `dimension` + `categoria`).

La participación se calcula como **suma de casuales ÷ suma de alquileres del grupo** (no como promedio de los `pct_casual` por hora, que daría el mismo peso a una hora de madrugada que a una de hora pico). `pct_casuales + pct_registrados = 100 %`.

In [24]:
partes = []
for dimension, columna in [("Tipo de día", "tipo_dia"), ("Franja horaria", "franja_horaria"),
                           ("Estación", "estacion"), ("Tipo de festivo", "tipo_festivo")]:
    parte = participacion(df_final, columna).rename(columns={columna: "categoria"})
    parte.insert(0, "dimension", dimension)
    partes.append(parte)
total = participacion(df_final.assign(categoria="Todo el período"), "categoria")
total.insert(0, "dimension", "Total")
partes.append(total)
p5 = pd.concat(partes, ignore_index=True)
p5["categoria"] = p5["categoria"].astype(str)
TABLAS["P5_participacion_casuales_y_registrados"] = p5
p5

,dimension,categoria,usuarios_casuales,usuarios_registrados,alquileres_total,pct_casuales,pct_registrados
0,Tipo de día,Laboral,303285,1989125,2292410,13.23,86.77
1,Tipo de día,Fin de semana,294373,627461,921834,31.93,68.07
2,Tipo de día,Festivo,22359,56076,78435,28.51,71.49
3,Franja horaria,Madrugada (0-5 h),19275,87234,106509,18.10,81.90
4,Franja horaria,Pico de la mañana (6-9 h),49273,580469,629742,7.82,92.18
5,Franja horaria,Mediodía (10-15 h),289201,716510,1005711,28.76,71.24
6,Franja horaria,Pico de la tarde (16-19 h),188055,913114,1101169,17.08,82.92
7,Franja horaria,Noche (20-23 h),74213,375335,449548,16.51,83.49
8,Estación,Invierno,60622,410726,471348,12.86,87.14
9,Estación,Primavera,203522,715067,918589,22.16,77.84


**Hallazgo.** En todo el período, los **registrados** representan el **81,17 %** de los alquileres y los **casuales** el **18,83 %**. Pero la proporción cambia según el momento:

- **Tipo de día:** los casuales son el 13,23 % en día laboral, 31,93 % en fin de semana y 28,51 % en festivo.
- **Franja:** el pico de la mañana es el más "registrado" (casuales 7,82 %) y el **mediodía** el que más casuales tiene (28,76 %); en el pico de la tarde son el 17,08 %.
- **Estación:** casuales 22,16 % en primavera y 21,31 % en verano, frente a 15,42 % en otoño y 12,86 % en invierno.
- **Tipo de festivo:** federales 29,76 % de casuales; el festivo local de D.C. 19,38 % (solo 2 fechas).

Es **consistente con** un uso de desplazamiento diario de los registrados (horas pico de días laborales) y un uso más recreativo de los casuales (mediodía, fines de semana, estaciones templadas), pero el dataset no dice para qué se usa cada alquiler.

**Utilidad.** Ayuda a segmentar: los registrados mueven la demanda de las horas pico laborales y los casuales la de fines de semana y mediodías de buen clima. **Limitaciones.** La participación se mide en alquileres, no en personas; no sabemos qué define a un usuario "casual" más allá de lo que documenta la fuente.

### P6. ¿Cuánto cambió la demanda entre 2011 y 2012?

**Variables:** `anio`, `usuarios_casuales`, `usuarios_registrados`, `alquileres_total`. **Nivel de detalle:** una fila por tipo de usuario.

Las horas observadas difieren entre años (las 165 horas sin registro no se reparten por igual), así que mostramos el **total** y también el **promedio por hora**, que no depende de cuántas horas haya.

In [25]:
filas = []
for etiqueta, columna in [("Casuales", "usuarios_casuales"), ("Registrados", "usuarios_registrados"), ("Total", "alquileres_total")]:
    t11 = int(df_final.loc[df_final["anio"] == 2011, columna].sum())
    t12 = int(df_final.loc[df_final["anio"] == 2012, columna].sum())
    h11 = int((df_final["anio"] == 2011).sum())
    h12 = int((df_final["anio"] == 2012).sum())
    filas.append({
        "tipo_usuario": etiqueta,
        "total_2011": t11, "total_2012": t12,
        "variacion_total_pct": (t12 - t11) / t11 * 100,
        "horas_observadas_2011": h11, "horas_observadas_2012": h12,
        "promedio_por_hora_2011": t11 / h11, "promedio_por_hora_2012": t12 / h12,
        "variacion_promedio_pct": ((t12 / h12) - (t11 / h11)) / (t11 / h11) * 100,
    })
p6 = pd.DataFrame(filas).round({"variacion_total_pct": 2, "promedio_por_hora_2011": 2, "promedio_por_hora_2012": 2, "variacion_promedio_pct": 2})
TABLAS["P6_variacion_2011_2012"] = p6
for anio in (2011, 2012):
    d = df_final[df_final["anio"] == anio]
    print(f"Participación de casuales en {anio}: {d['usuarios_casuales'].sum() / d['alquileres_total'].sum() * 100:.2f} %")
p6

Participación de casuales en 2011: 19.89 %
Participación de casuales en 2012: 18.19 %


,tipo_usuario,total_2011,total_2012,variacion_total_pct,horas_observadas_2011,horas_observadas_2012,promedio_por_hora_2011,promedio_por_hora_2012,variacion_promedio_pct
0,Casuales,247252,372765,50.76,8645,8734,28.60,42.68,49.23
1,Registrados,995851,1676811,68.38,8645,8734,115.19,191.99,66.66
2,Total,1243103,2049576,64.88,8645,8734,143.79,234.67,63.20


**Hallazgo.** Los alquileres totales pasaron de **1.243.103 (2011)** a **2.049.576 (2012)**: **+64,88 %**. Los casuales crecieron **+50,76 %** (247.252 a 372.765) y los registrados **+68,38 %** (995.851 a 1.676.811). El promedio por hora subió de 143,79 a 234,67 (**+63,20 %**; casuales +49,23 %, registrados +66,66 %). La variación del promedio es algo menor que la del total porque 2012 tiene 89 horas observadas más (8.734 contra 8.645). Como los registrados crecieron más, la participación de los casuales bajó de 19,89 % en 2011 a 18,19 % en 2012.

**Utilidad.** Permite anticipar que la demanda base del sistema aumentó de un año al otro y que el crecimiento vino sobre todo de usuarios registrados, por lo que la planificación no debería basarse solo en 2011.

**Limitaciones.** El dataset **no explica** el crecimiento (podría ser expansión del servicio, más socios u otros factores; no hay datos de cantidad de bicicletas ni de estaciones); son solo dos años, de una sola ciudad; y es descriptivo, no causal.

## 7. Exportación y verificación

Exportamos **por código** el dataset preparado, las tablas de cada pregunta y el `.txt` descriptivo del **DataFrame final** (el mismo `df_final` de las tablas). Las rutas son relativas al notebook. Después **releemos** cada archivo y comprobamos que coincide con lo que hay en memoria.

### 7.1 Dataset preparado y tablas `.csv`

In [26]:
RUTA_DATASET.parent.mkdir(parents=True, exist_ok=True)
RUTA_TABLAS.mkdir(parents=True, exist_ok=True)

df_final.to_csv(RUTA_DATASET, index=False, encoding="utf-8")
print(f"Dataset preparado: {RUTA_DATASET}  ({len(df_final)} filas x {df_final.shape[1]} columnas)")

for nombre, tabla in TABLAS.items():
    ruta = RUTA_TABLAS / f"{nombre}.csv"
    tabla.to_csv(ruta, index=False, encoding="utf-8")
    print(f"Tabla exportada  : {ruta}  ({len(tabla)} filas)")

Dataset preparado: ../datos/bike_sharing_preparado.csv  (17379 filas x 23 columnas)
Tabla exportada  : ../resultados/tablas/P1_demanda_por_franja_horaria.csv  (5 filas)
Tabla exportada  : ../resultados/tablas/P1_demanda_por_hora_del_dia.csv  (24 filas)
Tabla exportada  : ../resultados/tablas/P2_demanda_por_tipo_de_dia.csv  (3 filas)
Tabla exportada  : ../resultados/tablas/P2_demanda_por_tipo_de_dia_y_franja.csv  (15 filas)
Tabla exportada  : ../resultados/tablas/P2_demanda_por_festivo.csv  (11 filas)
Tabla exportada  : ../resultados/tablas/P3_demanda_por_estacion.csv  (4 filas)
Tabla exportada  : ../resultados/tablas/P3_demanda_por_mes.csv  (12 filas)
Tabla exportada  : ../resultados/tablas/P4_demanda_por_situacion_climatica.csv  (4 filas)
Tabla exportada  : ../resultados/tablas/P4_demanda_por_rango_de_temperatura.csv  (4 filas)
Tabla exportada  : ../resultados/tablas/P5_participacion_casuales_y_registrados.csv  (16 filas)
Tabla exportada  : ../resultados/tablas/P6_variacion_2011_2012.

### 7.2 Archivo `.txt` descriptivo del DataFrame final

Usamos `Reportes().guardar_reporte(...)` sobre `df_final`. El encabezado deja registrado de dónde salen los datos y la fuente de los festivos (LLM o respaldo).

In [27]:
n_filas = f"{len(df_final):,}".replace(",", ".")
encabezado = {
    "Dataset": loader.dataset_name,
    "Split": loader.split,
    "Revisión": loader.revision,
    "DataFrame descrito": "FINAL, transformado y enriquecido (el mismo que se exporta a CSV)",
    "Filas del dataset original": f"{len(df_original):,}".replace(",", "."),
    "Fuente de nombre_festivo y tipo_festivo": f"{enriquecedor.fuente} ({enriquecedor.detalle_fuente})",
    "Archivo del dataset preparado": str(RUTA_DATASET),
}
ruta_txt = Reportes().guardar_reporte(df_final, RUTA_RESUMEN, encabezado=encabezado)
print(f"Archivo generado: {ruta_txt}\n")
contenido = RUTA_RESUMEN.read_text(encoding="utf-8")
print(contenido)

Archivo generado: ../resultados/resumen_dataframe_final.txt

REPORTE DEL DATAFRAME
Dataset: t22000t/bike-sharing-tabular
Split: train
Revisión: 6b645e0cedbc5208fcca11ac94544b088478bf61
DataFrame descrito: FINAL, transformado y enriquecido (el mismo que se exporta a CSV)
Filas del dataset original: 17.379
Fuente de nombre_festivo y tipo_festivo: LLM (archivo versionado festivos_nemotron.json (modelo nvidia/nemotron-3-ultra-550b-a55b, generado el 2026-11-07T19:54:40+00:00); no se llamó al LLM)
Archivo del dataset preparado: ../datos/bike_sharing_preparado.csv
Filas: 17379
Columnas: 23
Duplicados: 0

Columnas y tipos:
- instant: int64
- fecha: datetime64[ns]
- fecha_hora: datetime64[ns]
- anio: int64
- mes: int64
- nombre_mes: category
- dia_semana: category
- hora: int64
- franja_horaria: category
- tipo_dia: category
- nombre_festivo: object
- tipo_festivo: object
- estacion: category
- clima: category
- temp_c: float64
- sensacion_c: float64
- rango_temperatura: category
- humedad_pct:

### 7.3 Verificación de lo exportado

Comprobamos que: (1) el `.txt` corresponde al DataFrame final; (2) el CSV del dataset coincide con `df_final`; (3) cada CSV de tabla coincide con su tabla en memoria; (4) los totales y porcentajes son consistentes.

In [28]:
import numpy as np

# (1) El .txt corresponde al DataFrame final
assert f"Filas: {len(df_final)}" in contenido
assert f"Columnas: {len(df_final.columns)}" in contenido
assert all(f"- {c}: " in contenido for c in df_final.columns)
for c, t in df_final.dtypes.items():
    assert f"- {c}: {t}" in contenido
for c, n in df_final.isna().sum().items():
    assert f"- {c}: {n}" in contenido
print("1. El .txt coincide con el DataFrame final (filas, columnas, tipos y nulos).")


def mismo_contenido(memoria, leido):
    assert list(memoria.columns) == list(leido.columns), "Las columnas no coinciden"
    assert len(memoria) == len(leido), "La cantidad de filas no coincide"
    for c in memoria.columns:
        a, b = memoria[c], leido[c]
        if pd.api.types.is_numeric_dtype(a):
            assert np.allclose(a.astype(float), b.astype(float)), f"Difieren valores en {c}"
        elif pd.api.types.is_datetime64_any_dtype(a):
            assert (pd.to_datetime(b) == a).all(), f"Difieren fechas en {c}"
        else:
            assert (a.astype(str).values == b.astype(str).values).all(), f"Difieren textos en {c}"


# (2) Dataset preparado
leido = pd.read_csv(RUTA_DATASET)
mismo_contenido(df_final, leido)
assert int(leido["alquileres_total"].sum()) == TOTAL_ALQUILERES
total_fmt = f"{TOTAL_ALQUILERES:,}".replace(",", ".")
print(f"2. {RUTA_DATASET.name} coincide con df_final ({len(leido)} filas, suma de alquileres {total_fmt}).")

# (3) y (4) Tablas: releer, comparar y revisar totales y porcentajes
verificacion = []
for nombre, tabla in TABLAS.items():
    releida = pd.read_csv(RUTA_TABLAS / f"{nombre}.csv")
    mismo_contenido(tabla.astype({c: str for c in tabla.select_dtypes("category").columns}), releida)
    fila = {"tabla": nombre, "filas": len(releida), "coincide_con_memoria": True}
    if "total_alquileres" in releida.columns and nombre != "P2_demanda_por_festivo" and "dimension" not in releida.columns:
        fila["suma_total_alquileres"] = int(releida["total_alquileres"].sum())
        assert fila["suma_total_alquileres"] == TOTAL_ALQUILERES, f"{nombre}: los totales no suman el total general"
    if "pct_del_total_alquileres" in releida.columns:
        fila["suma_pct"] = round(float(releida["pct_del_total_alquileres"].sum()), 2)
        assert abs(fila["suma_pct"] - 100) <= 0.005 * len(releida) + 1e-9, f"{nombre}: los porcentajes no suman 100"
    verificacion.append(fila)

# P2 por festivo: la suma debe ser igual a los alquileres de las horas festivas
leido_fest = pd.read_csv(RUTA_TABLAS / "P2_demanda_por_festivo.csv")
assert int(leido_fest["total_alquileres"].sum()) == int(df_final.loc[df_final["tipo_dia"] == "Festivo", "alquileres_total"].sum())
assert int(leido_fest["horas_observadas"].sum()) == int((df_final["tipo_dia"] == "Festivo").sum())

# P5: en cada dimensión los totales suman el total general y casuales + registrados = 100 %
p5_leida = pd.read_csv(RUTA_TABLAS / "P5_participacion_casuales_y_registrados.csv")
for dimension, grupo in p5_leida.groupby("dimension"):
    assert int(grupo["alquileres_total"].sum()) == TOTAL_ALQUILERES, dimension
    assert (grupo["usuarios_casuales"] + grupo["usuarios_registrados"] == grupo["alquileres_total"]).all(), dimension
assert np.allclose(p5_leida["pct_casuales"] + p5_leida["pct_registrados"], 100, atol=0.011)

# P6: los totales por año suman el total general
p6_leida = pd.read_csv(RUTA_TABLAS / "P6_variacion_2011_2012.csv").set_index("tipo_usuario")
assert p6_leida.loc["Total", ["total_2011", "total_2012"]].sum() == TOTAL_ALQUILERES
assert (p6_leida.loc["Casuales", ["total_2011", "total_2012"]].sum() + p6_leida.loc["Registrados", ["total_2011", "total_2012"]].sum()) == TOTAL_ALQUILERES

print(f"3. Las {len(TABLAS)} tablas exportadas coinciden con las tablas en memoria.")
print("4. Totales (suma de alquileres) y porcentajes (suman 100 %) consistentes en todas las tablas.")
pd.DataFrame(verificacion).fillna("-")

1. El .txt coincide con el DataFrame final (filas, columnas, tipos y nulos).


2. bike_sharing_preparado.csv coincide con df_final (17379 filas, suma de alquileres 3.292.679).


3. Las 11 tablas exportadas coinciden con las tablas en memoria.
4. Totales (suma de alquileres) y porcentajes (suman 100 %) consistentes en todas las tablas.


,tabla,filas,coincide_con_memoria,suma_total_alquileres,suma_pct
0,P1_demanda_por_franja_horaria,5,True,3292679.0,99.99
1,P1_demanda_por_hora_del_dia,24,True,3292679.0,99.99
2,P2_demanda_por_tipo_de_dia,3,True,3292679.0,100.0
3,P2_demanda_por_tipo_de_dia_y_franja,15,True,3292679.0,99.99
4,P2_demanda_por_festivo,11,True,-,-
5,P3_demanda_por_estacion,4,True,3292679.0,100.01
6,P3_demanda_por_mes,12,True,3292679.0,100.01
7,P4_demanda_por_situacion_climatica,4,True,3292679.0,100.0
8,P4_demanda_por_rango_de_temperatura,4,True,3292679.0,100.0
9,P5_participacion_casuales_y_registrados,16,True,-,-


### 7.4 Archivos generados

In [29]:
archivos = [RUTA_DATASET, RUTA_RESUMEN, RUTA_LLM, *sorted(RUTA_TABLAS.glob("*.csv"))]
pd.DataFrame({
    "archivo": [str(a) for a in archivos],
    "existe": [a.exists() for a in archivos],
    "tamaño_kb": [round(a.stat().st_size / 1024, 1) if a.exists() else None for a in archivos],
})

,archivo,existe,tamaño_kb
0,../datos/bike_sharing_preparado.csv,True,3418.2
1,../resultados/resumen_dataframe_final.txt,True,3.1
2,../llm/festivos_nemotron.json,True,4.0
3,../resultados/tablas/P1_demanda_por_franja_hor...,True,0.4
4,../resultados/tablas/P1_demanda_por_hora_del_d...,True,1.3
5,../resultados/tablas/P2_demanda_por_festivo.csv,True,0.7
6,../resultados/tablas/P2_demanda_por_tipo_de_di...,True,0.2
7,../resultados/tablas/P2_demanda_por_tipo_de_di...,True,1.0
8,../resultados/tablas/P3_demanda_por_estacion.csv,True,0.3
9,../resultados/tablas/P3_demanda_por_mes.csv,True,0.6


## 8. Conclusiones del avance


**Qué permite conocer el avance.**

- El dataset quedó **validado** (20 controles, 0 problemas de calidad) y **transformado** sin perder filas: 17.379 filas, 23 columnas, 0 nulos. Faltan 165 horas del calendario que **no se imputaron**, y quedaron documentados dos hallazgos informativos (humedad 0 en 22 filas del 2011-03-10 y viento 0 en 2.180 filas).
- Se **enriqueció** con variables derivadas (franja horaria, tipo de día, rango de temperatura, `pct_casual`, clima en unidades originales) y con el **nombre y tipo de festivo** (LLM o respaldo OPM según la ejecución), sin multiplicar filas ni cambiar la suma de alquileres.
- Se respondieron P1 a P6: la demanda se concentra en las **franjas pico de la tarde (33,44 % del total) y de la mañana**; los días laborales tienen dos picos y los fines de semana y festivos un pico al mediodía; el invierno tiene menos de la mitad de la demanda del verano (111,11 contra 236,02 por hora); la lluvia ligera reduce el promedio a 111,58 y el calor lo aumenta; los casuales son el 18,83 % y pesan más en fines de semana y mediodías; y la demanda total creció **64,88 %** de 2011 a 2012.

**Qué queda sin resolver.**

- **Causas:** todo es descriptivo; no sabemos por qué cambia la demanda ni cuánto del efecto se debe al clima, la estación o la hora (están mezclados).
- **Dónde:** sin datos por estación de bicicletas ni por viaje no se puede decidir *dónde* redistribuir.
- **Calidad pendiente:** la causa de las 165 horas faltantes, la unidad del viento y los ceros de humedad y de viento siguen sin documentarse.
- **Festivos:** Nemotron (`nvidia/nemotron-3-ultra-550b-a55b`) clasificó las 21 fechas festivas eligiendo de una lista cerrada de nombres; las 21 coinciden en fecha, nombre y tipo con el calendario oficial de la OPM. La respuesta quedó guardada en `AE3/llm/festivos_nemotron.json`, así que las siguientes ejecuciones (incluido el bot) no vuelven a llamar al LLM.
- **Siguiente entrega (AE4):** visualizar estos resultados con gráficos que respeten el orden de las categorías ya definidas.


### 8.1 Diccionario de variables del DataFrame final

In [30]:
diccionario = pd.DataFrame([
    ("instant", "Original", "Identificador de la hora de servicio (índice del dataset)", "entero", "-"),
    ("fecha", "Incorporada", "Día calendario (de `dteday`)", "fecha", "fecha"),
    ("fecha_hora", "Incorporada", "Momento exacto de la hora de servicio", "fecha-hora", "fecha-hora"),
    ("anio", "Incorporada", "Año (de `yr`): 2011 o 2012", "entero", "año"),
    ("mes", "Original (`mnth`)", "Número del mes (1-12)", "entero", "-"),
    ("nombre_mes", "Incorporada", "Nombre del mes", "categoría ordenada", "-"),
    ("dia_semana", "Incorporada", "Día de la semana (de `weekday`)", "categoría ordenada", "-"),
    ("hora", "Original (`hr`)", "Hora del día (0-23)", "entero", "hora"),
    ("franja_horaria", "Incorporada", "Franja del día según el perfil de demanda", "categoría ordenada", "-"),
    ("tipo_dia", "Incorporada", "Laboral, fin de semana o festivo", "categoría ordenada", "-"),
    ("nombre_festivo", "Incorporada (LLM o respaldo)", "Nombre del festivo; `No festivo` si no lo es", "texto", "-"),
    ("tipo_festivo", "Incorporada (LLM o respaldo)", "Federal o local de D.C.; `No festivo` si no lo es", "texto", "-"),
    ("estacion", "Incorporada", "Estación del año (de `season`)", "categoría ordenada", "-"),
    ("clima", "Incorporada", "Situación climática (de `weathersit`)", "categoría ordenada", "-"),
    ("temp_c", "Incorporada", "Temperatura (de `temp`)", "decimal", "°C"),
    ("sensacion_c", "Incorporada", "Sensación térmica (de `atemp`)", "decimal", "°C"),
    ("rango_temperatura", "Incorporada", "Rango de temperatura", "categoría ordenada", "°C"),
    ("humedad_pct", "Incorporada", "Humedad relativa (de `hum`)", "decimal", "%"),
    ("viento_original", "Incorporada", "Velocidad del viento (de `windspeed`)", "decimal", "sin documentar"),
    ("usuarios_casuales", "Original (`casual`)", "Alquileres de usuarios casuales en la hora", "entero", "alquileres"),
    ("usuarios_registrados", "Original (`registered`)", "Alquileres de usuarios registrados en la hora", "entero", "alquileres"),
    ("alquileres_total", "Original (`cnt`)", "Total de alquileres en la hora (casuales + registrados)", "entero", "alquileres"),
    ("pct_casual", "Incorporada", "Porcentaje de alquileres casuales en esa hora", "decimal", "%"),
], columns=["variable", "origen", "significado", "tipo", "unidad"])
assert set(diccionario["variable"]) == set(df_final.columns) and len(diccionario) == df_final.shape[1]
diccionario

,variable,origen,significado,tipo,unidad
0,instant,Original,Identificador de la hora de servicio (índice d...,entero,-
1,fecha,Incorporada,Día calendario (de `dteday`),fecha,fecha
2,fecha_hora,Incorporada,Momento exacto de la hora de servicio,fecha-hora,fecha-hora
3,anio,Incorporada,Año (de `yr`): 2011 o 2012,entero,año
4,mes,Original (`mnth`),Número del mes (1-12),entero,-
5,nombre_mes,Incorporada,Nombre del mes,categoría ordenada,-
6,dia_semana,Incorporada,Día de la semana (de `weekday`),categoría ordenada,-
7,hora,Original (`hr`),Hora del día (0-23),entero,hora
8,franja_horaria,Incorporada,Franja del día según el perfil de demanda,categoría ordenada,-
9,tipo_dia,Incorporada,"Laboral, fin de semana o festivo",categoría ordenada,-
